In [0]:
from pyspark.sql import functions as F

CATALOG = "automotive_project"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

# Make sure Silver schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

results = []

for table in tables:
    try:
        df = spark.table(f"{BRONZE}.{table}")

        # Remove duplicate records
        df = df.dropDuplicates()

        # Clean whitespace from all string columns
        for col_name, data_type in df.dtypes:
            if data_type == "string":
                df = df.withColumn(
                    col_name,
                    F.trim(F.col(col_name))
                )

        # Write cleaned data to Silver as Delta
        (
            df.write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(f"{SILVER}.{table}")
        )

        results.append(
            (table, df.count(), len(df.columns), "SUCCESS")
        )

        print(f"✓ {table} → Silver")

    except Exception as e:
        results.append(
            (table, 0, 0, f"FAILED: {str(e)[:200]}")
        )

        print(f"✗ {table} → {str(e)[:200]}")

print("\n===== SILVER TRANSFORMATION COMPLETE =====")

display(
    spark.createDataFrame(
        results,
        ["table_name", "row_count", "column_count", "status"]
    )
)

✓ addresses → Silver
✓ customer_vehicles → Silver
✓ customers → Silver
✓ dealers → Silver
✓ parts_catalog → Silver
✓ parts_used → Silver
✓ service_appointments → Silver
✓ service_centers → Silver
✓ service_order_tasks → Silver
✓ service_orders → Silver
✓ service_types → Silver
✓ technicians → Silver
✓ vehicle_models → Silver
✓ vehicles → Silver
✓ warranty_claims → Silver

===== SILVER TRANSFORMATION COMPLETE =====


table_name,row_count,column_count,status
addresses,10100,7,SUCCESS
customer_vehicles,10000,8,SUCCESS
customers,10000,8,SUCCESS
dealers,20,6,SUCCESS
parts_catalog,50,6,SUCCESS
parts_used,500000,7,SUCCESS
service_appointments,10000,7,SUCCESS
service_centers,50,6,SUCCESS
service_order_tasks,500000,6,SUCCESS
service_orders,10000,8,SUCCESS
